# Mind the Gap (MTG-AD): tables and figures from per-video metrics

Aggregates per-video metric JSONs (`output_{VIDEO_ID}.json`) into the paper's
tables and figures. By default it reads the released metrics of the
Hugging Face dataset (`raw/test/model_metrics/{config}/`); point `CONFIG_DIRS`
at `run_ad_eval.py` output folders (`{output_dir}/metrics/`) to score your own
systems the same way.

Aggregation (see `aggregate.py`):

* On-Screen Text only counts videos with on-screen text; Timeliness, Overlap and
  Avg. Silence values longer than the video are dropped.
* Rows listed in `raw/test/scoring_exclusions.json` are left out (the 25
  unstandardized Qwen-3.5-Omni-Flash responses).
* The prompt comparison uses only the videos that have a metrics file for
  **every** prompt. The model comparisons (tables and figures) score each model
  on **all** test videos it has output for (`common_videos=False`): 101 for
  most models, fewer for Qwen-3.5-Omni-Flash. Cells are mean ± 95% CI
  (1.96·SEM).

> The released metrics cover the 101 **test** videos. Some paper tables also
> include validation videos (e.g. the prompt comparison uses 372 videos), so
> values computed here can differ slightly from the paper.

Requires the packages in `requirements.txt` (incl. `matplotlib` and `seaborn`).

In [ ]:
import os
import pandas as pd
import aggregate as agg

pd.set_option("display.width", 200)

# Root of the Hugging Face dataset download.
DATASET_DIR = "../mind-the-gap-inline-ad"
METRICS_ROOT = os.path.join(DATASET_DIR, "raw/test/model_metrics")
EXCLUSIONS_FILE = os.path.join(DATASET_DIR, "raw/test/scoring_exclusions.json")
OUT_DIR = "paper_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

CONFIGS = [
    "gemini_2p5_flash_simple", "gemini_2p5_flash_guideline",
    "gemini_2p5_flash_guideline_asr", "gemini_2p5_flash_timing_asr",
    "qwen_vl_max", "gpt_4_1_mini", "claude_4", "claude_4_5", "gpt_5_2",
    "gemini_3p5_flash_timing_asr", "qwen3p5_omni_flash", "qwen3p5_omni_plus",
]
# Configuration -> per-video metrics directory. Add your own systems here, e.g.
# CONFIG_DIRS["my_model"] = "eval_results/my_model/metrics".
CONFIG_DIRS = {c: os.path.join(METRICS_ROOT, c) for c in CONFIGS}

LABELS = {
    "gemini_2p5_flash_simple": "Simple AD",
    "gemini_2p5_flash_guideline": "Guideline only",
    "gemini_2p5_flash_guideline_asr": "Guideline + ASR",
    "gemini_2p5_flash_timing_asr": "Gemini-2.5 Flash",
    "qwen_vl_max": "Qwen-2-VL-Max",
    "gpt_4_1_mini": "GPT-4.1 mini",
    "claude_4": "Claude 4",
    "claude_4_5": "Claude 4.5",
    "gpt_5_2": "GPT-5.2",
    "gemini_3p5_flash_timing_asr": "Gemini-3.5 Flash$^{\\dagger}$",
    "qwen3p5_omni_flash": "Qwen-3.5-Omni-Flash$^{\\dagger}$",
    "qwen3p5_omni_plus": "Qwen-3.5-Omni-Plus$^{\\dagger}$",
}
PLOT_LABELS = {k: v.replace("$^{\\dagger}$", "†") for k, v in LABELS.items()}

In [ ]:
#@title Load per-video metrics
exclusions = agg.load_exclusions(EXCLUSIONS_FILE)
df = agg.load_configs(CONFIG_DIRS, exclusions=exclusions)
VIDEOA11Y_LABELS = {m: agg.METRICS[m][0].replace("-Ref", "")
                    for m in agg.VIDEOA11Y_METRICS}

## Prompt comparison (paper Table `tab:main_results`)

Gemini 2.5 Flash with the four prompting strategies. The paper's table also has
a Human AD (GT) column; add its metrics folder to `CONFIG_DIRS` to include it.

In [ ]:
PROMPTS = ["gemini_2p5_flash_simple", "gemini_2p5_flash_guideline",
           "gemini_2p5_flash_guideline_asr", "gemini_2p5_flash_timing_asr"]
prompt_labels = {c: LABELS[c] for c in PROMPTS}
prompt_labels["gemini_2p5_flash_timing_asr"] = "Gd. + Timing ASR"

prompt_summary = agg.summarize(df, PROMPTS)
print("Common videos:", prompt_summary.attrs["num_common_videos"])
print(agg.latex_table(prompt_summary, prompt_labels,
                      metric_labels=VIDEOA11Y_LABELS))

In [ ]:
FIG_METRICS = ["metric_G1.1_precision_avg_ad_in_visual_info",
               "metric_1_2_narrates_onscreen_text", "metric_g_2_avg_diff_time",
               "total_overlap", "avg_silence_duration_w_ad_tts"]
FIG_TITLES = {
    "metric_G1.1_precision_avg_ad_in_visual_info": "Prioritization",
    "metric_1_2_narrates_onscreen_text": "On-Screen Text",
    "metric_g_2_avg_diff_time": "Timeliness (s)",
    "total_overlap": "Audio Overlaps (s)",
    "avg_silence_duration_w_ad_tts": "Avg. Silence Duration (s)",
}
_ = agg.prompt_figure(
    df, PROMPTS, prompt_labels, FIG_METRICS, FIG_TITLES,
    ylims={"metric_g_2_avg_diff_time": 15, "total_overlap": 65,
           "avg_silence_duration_w_ad_tts": 25},
    out_path=f"{OUT_DIR}/prompt_comparison_bib.png")

## Model comparison: initial frontier models (paper Table `tab:main_res_model_comp`)

Each model is scored on all test videos it has output for.

In [ ]:
INITIAL_MODELS = ["gemini_2p5_flash_timing_asr", "qwen_vl_max", "gpt_4_1_mini",
                  "claude_4", "claude_4_5", "gpt_5_2"]
initial_summary = agg.summarize(df, INITIAL_MODELS, common_videos=False)
print("Videos per model:", initial_summary.attrs["num_videos_per_config"])
print(agg.latex_table(initial_summary, {c: LABELS[c] for c in INITIAL_MODELS},
                      metric_labels=VIDEOA11Y_LABELS))

## Model comparison: next-generation omni-modal models (paper Table `tab:rebuttal_models_comp`)

Each model is scored on all test videos it has output for (Qwen-3.5-Omni-Flash
has fewer than the others).
† models were judged with Gemini 3.5 Flash for all LLM-judged metrics
(G1.1, G1.2, G2, VideoA11y-Ref).

In [ ]:
OMNI_MODELS = ["gemini_2p5_flash_timing_asr", "gemini_3p5_flash_timing_asr",
               "qwen_vl_max", "qwen3p5_omni_flash", "qwen3p5_omni_plus"]
omni_summary = agg.summarize(df, OMNI_MODELS, common_videos=False)
print("Videos per model:", omni_summary.attrs["num_videos_per_config"])
print(agg.latex_table(omni_summary, {c: LABELS[c] for c in OMNI_MODELS}))

## Model comparison figures (paper Figures `model_comparisons_main5_bib`, `modelcomp2_video_a11y_bib`)

In [ ]:
FIG_MODELS = ["gpt_4_1_mini", "gpt_5_2", "claude_4_5", "qwen_vl_max",
              "qwen3p5_omni_flash", "qwen3p5_omni_plus",
              "gemini_2p5_flash_timing_asr", "gemini_3p5_flash_timing_asr"]
HATCHES = ["", "///", "", "\\", "", ".."]
print("Videos per model:",
      agg.select_videos(df, FIG_MODELS, common_videos=False)
      .groupby("config").video_id.nunique().to_dict())
_ = agg.bar_figure(
    df, FIG_MODELS, PLOT_LABELS, FIG_METRICS,
    {"metric_G1.1_precision_avg_ad_in_visual_info": "G1.1 - Prioritization",
     "metric_1_2_narrates_onscreen_text": "G1.3 - On-Screen Text",
     "metric_g_2_avg_diff_time": "G2 - Timeliness (s)",
     "total_overlap": "G7 - Overlaps w. TTS (s)",
     "avg_silence_duration_w_ad_tts": "Avg. Silence Duration (s)"},
    ylims={"metric_G1.1_precision_avg_ad_in_visual_info": 1.1,
           "metric_g_2_avg_diff_time": 50, "total_overlap": 80,
           "avg_silence_duration_w_ad_tts": 35},
    hatches=HATCHES, drop_timeliness_outliers=True, common_videos=False,
    label_rotation=45,
    out_path=f"{OUT_DIR}/model_comparisons_main5_bib.png")

In [ ]:
_ = agg.bar_figure(
    df, FIG_MODELS, PLOT_LABELS, agg.VIDEOA11Y_METRICS,
    {"videoa11y_descriptive": "(VideoA11y) Descriptive",
     "videoa11y_objective": "(VideoA11y) Objective",
     "videoa11y_accurate": "(VideoA11y) Accurate",
     "videoa11y_clear": "(VideoA11y) Clear"},
    ylims={m: 5.5 for m in agg.VIDEOA11Y_METRICS},
    ylabel="Mean Score [1-5] (95% CI)", hatches=HATCHES, sharey=True,
    common_videos=False, label_rotation=45,
    out_path=f"{OUT_DIR}/modelcomp2_video_a11y_bib.png")